In [0]:
%run ./ingestion_utils

In [0]:
# DataUSA API configuration

DATAUSA_URL = (
    "https://honolulu-api.datausa.io/tesseract/data.jsonrecords"
    "?cube=acs_yg_total_population_1"
    "&drilldowns=Year%2CNation"
    "&locale=en"
    "&measures=Population"
)

VOLUME_PATH = "/Volumes/workspace/rearc/rearc_landing"

DATAUSA_RAW_PATH = f"{VOLUME_PATH}/raw/datausa"
DATAUSA_FILE_PATH = f"{DATAUSA_RAW_PATH}/population.json"
DATAUSA_MANIFEST_PATH = f"{DATAUSA_RAW_PATH}/manifest.json"

# Create landing directory if it does not exist
dbutils.fs.mkdirs(DATAUSA_RAW_PATH)

print("DataUSA ingestion configuration loaded")
print("Source:", DATAUSA_URL)
print("Landing folder:", DATAUSA_RAW_PATH)
print("Output file:", DATAUSA_FILE_PATH)
print("Manifest:", DATAUSA_MANIFEST_PATH)

In [0]:
# Download DataUSA population data

session = create_http_session()

result = download_file(
    session,
    DATAUSA_URL,
    DATAUSA_FILE_PATH
)

print("DATAUSA DOWNLOAD RESULT")
print("File size:", result["file_size"])
print("SHA-256:", result["sha256"])
print("Saved path:", result["file_path"])

assert result["file_size"] > 0, "Downloaded file is empty"

print("DataUSA download test: PASSED")

In [0]:
import json

# Read the downloaded JSON from Unity Catalog Volume
raw_json = dbutils.fs.head(DATAUSA_FILE_PATH, 1000000)

# Parse JSON
population_data = json.loads(raw_json)

# Validate root structure
assert isinstance(population_data, dict), "Expected JSON object"

print("JSON validation: PASSED")
print("Root keys:", list(population_data.keys()))

# Inspect records
records = population_data.get("data", [])

print("Record count:", len(records))
print("First record:")

if records:
    print(records[0])

# Inspect columns if available
print("Columns:", population_data.get("columns"))

In [0]:
# Validate DataUSA population records

records = population_data["data"]

years = [int(row["Year"]) for row in records]

assert len(records) > 0, "No population records found"

assert all(row["Nation"] == "United States" for row in records), \
    "Unexpected nation found"

assert all(row["Population"] is not None for row in records), \
    "Null population found"

assert len(years) == len(set(years)), \
    "Duplicate years found"

print("Total records:", len(records))
print("Year range:", min(years), "-", max(years))
print("Unique years:", sorted(years))
print("All records belong to United States")
print("No null population values")
print("No duplicate years")
print("DATAUSA DATA QUALITY CHECKS: PASSED")

In [0]:
# Inspect all population records

for row in sorted(records, key=lambda x: int(x["Year"])):
    print(
        "Year:", row["Year"],
        "| Nation:", row["Nation"],
        "| Population:", row["Population"]
    )

# Identify missing years
expected_years = set(range(min(years), max(years) + 1))
missing_years = sorted(expected_years - set(years))

print("\nMissing years:", missing_years)

In [0]:
print("DATAUSA INGESTION VALIDATION")
print("----------------------------")
print("Total records:", len(records))
print("Available years:", sorted(years))
print("Missing years:", missing_years)
print("Source gap explanation:")
print("2020 standard ACS 1-year estimates were not released.")
print("No artificial population value has been added.")

assert len(records) == 11
assert missing_years == [2020]

print("\nDATAUSA VALIDATION COMPLETED")